# 10 - Micro Patterns

Hour-of-day by day-of-week activity heatmaps, weekend vs weekday
comparisons, and monthly PR volume heatmaps across repos.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from oss_pulse.visualize.heatmaps import (
    plot_activity_heatmap,
    plot_correlation_matrix,
    plot_monthly_heatmap,
)
from oss_pulse.visualize.style import PALETTE, setup_style

setup_style()

In [ ]:
# Load featured PR data and monthly aggregation
DATA_DIR = Path("../data/processed")

pr_df = pd.read_parquet(DATA_DIR / "pr_events_featured.parquet")
monthly_df = pd.read_parquet(DATA_DIR / "repo_monthly.parquet")

print(f"PR events: {len(pr_df)} rows")
print(f"Columns with temporal features: {[c for c in pr_df.columns if c in ['hour', 'day_of_week', 'is_weekend', 'month']]}")

In [ ]:
# Hour x Day-of-Week activity heatmap (all repos)
# TODO: run with real data
fig = plot_activity_heatmap(pr_df, title="PR Activity: All Repos")
fig.show()

In [ ]:
# Weekend vs Weekday comparison
# TODO: run with real data
weekend_stats = pr_df.groupby("is_weekend").agg(
    pr_count=("pr_number", "nunique"),
    unique_authors=("author", "nunique"),
    median_merge_hours=("time_to_merge_hours", "median"),
).reset_index()
weekend_stats["label"] = weekend_stats["is_weekend"].map({True: "Weekend", False: "Weekday"})

print("Weekend vs Weekday:")
print(weekend_stats[["label", "pr_count", "unique_authors", "median_merge_hours"]])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [PALETTE["primary"], PALETTE["warning"]]

for i, metric in enumerate(["pr_count", "unique_authors", "median_merge_hours"]):
    axes[i].bar(weekend_stats["label"], weekend_stats[metric], color=colors, edgecolor=PALETTE["bg"])
    axes[i].set_title(metric.replace("_", " ").title(), fontsize=12, fontweight="bold")

plt.suptitle("Weekend vs Weekday", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Per-repo activity heatmaps (top 3 repos by volume)
# TODO: run with real data
top3_repos = (
    pr_df.groupby("repo_name")["pr_number"]
    .nunique()
    .nlargest(3)
    .index.tolist()
)

for repo in top3_repos:
    repo_pr = pr_df[pr_df["repo_name"] == repo]
    fig = plot_activity_heatmap(repo_pr, title=f"Activity Heatmap: {repo}")
    fig.show()

In [ ]:
# Monthly PR volume heatmap across repos
# TODO: run with real data
fig = plot_monthly_heatmap(
    monthly_df,
    repos=top3_repos,
    title="Monthly PR Volume: Top 3 Repos",
)
fig.show()

In [ ]:
# Monthly seasonality: average PR count by calendar month
# TODO: run with real data
month_avg = pr_df.groupby("month")["pr_number"].nunique().reset_index()
month_avg.columns = ["month", "pr_count"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    month_avg["month"], month_avg["pr_count"],
    color=PALETTE["primary"], edgecolor=PALETTE["bg"],
)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_title("PR Count by Calendar Month", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Unique PRs")
plt.tight_layout()
plt.show()